In [ ]:
import copy
import functools
import glob
import math
import pickle
import scipy
import sklearn
import warnings

warnings.filterwarnings('ignore')

import itertools
import graphviz as gr
import numpy as np
import os
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import seaborn as sns


from matplotlib import style
from matplotlib import pyplot as plt
from pprint import pprint
from scipy import stats, special
from sklearn import datasets, mixture
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

from scipy.stats import ttest_ind


from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)

import plotly.express as px
import plotly.figure_factory as ff

%matplotlib inline

style.use("fivethirtyeight")
pd.set_option("display.max_columns", 6)

np.random.seed(0)
pd.set_option('display.max_columns', None)

import dask
import dask.dataframe as dd
from dask.distributed import Client


In [ ]:
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)

### Binary Classification

In [ ]:
def binary_classification_performance(df, date_col="date", outcome_col="delta_ranked", predictor_col="TLGRF_decision"):
    # Compute TP, TN, FP, FN
    selected_df = df[df[predictor_col]!=0]
    
    not_selected_df = df[df[predictor_col]==0]
    
    Results_Thresh = pd.DataFrame()
    #Total
    Total_Decision_Set = df.groupby(date_col)[outcome_col].count()
    Results_Thresh["Total"] = Total_Decision_Set
    #P
    #P = df.groupby(date_col)["capacity"].max()
    
    #TP
    TP = selected_df.groupby(date_col).apply(lambda x: (x[outcome_col] <= x["capacity"]).sum())
    Results_Thresh["TP"] = TP
    #FP = FN
    FP = selected_df.groupby(date_col).apply(lambda x: (x[outcome_col] > x["capacity"]).sum())
    Results_Thresh["FP"] = FP
    
    FN = not_selected_df.groupby(date_col).apply(lambda x: (x[outcome_col] <= x["capacity"]).sum())
    Results_Thresh["FN"] = FN
    #N
    TN = not_selected_df.groupby(date_col).apply(lambda x: (x[outcome_col] > x["capacity"]).sum())
    Results_Thresh["TN"] = TN
    
    Results_Thresh["P"] = TP + FN
    Results_Thresh["N"] = TN + FP
    #Results_Thresh["TN"] = Results_Thresh["N"] - Results_Thresh["FP"]
    #TN
    Results_Thresh = Results_Thresh[["Total", "P", "N", "TP", "FP", "FN", "TN"]]
    
    TP = Results_Thresh.sum()["TP"]
    FP = Results_Thresh.sum()["FP"]
    FN = Results_Thresh.sum()["FN"]
    TN = Results_Thresh.sum()["TN"]


    PPV = Results_Thresh.sum()["TP"]/(Results_Thresh.sum()["TP"] + Results_Thresh.sum()["FP"])
    FPV = Results_Thresh.sum()["FP"]/(Results_Thresh.sum()["TN"] + Results_Thresh.sum()["FP"])
    TPR = Results_Thresh.sum()["TP"]/(Results_Thresh.sum()["TP"] + Results_Thresh.sum()["FN"])
    TNR = Results_Thresh.sum()["TN"]/(Results_Thresh.sum()["TN"] + Results_Thresh.sum()["FP"])

    Precision = Results_Thresh.sum()["TP"] / (Results_Thresh.sum()["TP"] + Results_Thresh.sum()["FP"])
    Recall = Results_Thresh.sum()["TP"] / (Results_Thresh.sum()["TP"] + Results_Thresh.sum()["FN"])
    F1 = 2 * Precision * Recall / (Precision + Recall)

    # create a pandas dataframe from the confusion matrix values
    confusion_table = pd.DataFrame({'actual_positive': [1, 1, 0, 0], 'predicted_positive': [0, 1, 0, 1], 'count': [FN, TP, TN, FP]})

    # create a pivot table to represent the confusion matrix
    confusion_matrix = pd.pivot_table(confusion_table, values='count', index=['actual_positive'], columns=['predicted_positive'])

    # set the column and index names of the pivot table
    confusion_matrix.columns.name = 'Predicted Label'
    confusion_matrix.index.name = 'Actual Label'
    #confusion_matrix = confusion_matrix.T


    print("PPV={}, FPV={}\nTPR={}, TNR={}\nF1={}".format(PPV, FPV, TPR, TNR, F1))
    print(Results_Thresh.sum())
    confusion_matrix
    
    return Results_Thresh, (PPV, FPV, TPR, TNR, F1), confusion_matrix

In [ ]:
def binary_classification_performance_counter(df, date_col="date", outcome_col="delta_ranked", predictor="TLGRF"):
    # Compute TP, TN, FP, FN
    
    assert predictor in {"CDPHE", "TLGRF"}
        
    Results_Thresh = pd.DataFrame()
    #Total
    Total_Decision_Set = df.groupby(date_col)[outcome_col].count()
    Results_Thresh["Total"] = Total_Decision_Set
    #P
    #P = df.groupby(date_col)["capacity"].max()
    
    #TP
    Results_Thresh["TP"] = df.groupby(date_col)[predictor + "_TP"].sum()
    #FP
    Results_Thresh["FP"] = df.groupby(date_col)[predictor + "_FP"].sum()
    #FN
    Results_Thresh["FN"] = df.groupby(date_col)[predictor + "_FN"].sum()
    #TN
    Results_Thresh["TN"] = df.groupby(date_col)[predictor + "_TN"].sum()
    
    Results_Thresh["P"] = Results_Thresh["TP"] + Results_Thresh["FN"]
    Results_Thresh["N"] = Results_Thresh["TN"] + Results_Thresh["FP"]
    #Results_Thresh["TN"] = Results_Thresh["N"] - Results_Thresh["FP"]
    #TN
    Results_Thresh = Results_Thresh[["Total", "P", "N", "TP", "FP", "FN", "TN"]]
    
    TP = Results_Thresh.sum()["TP"]
    FP = Results_Thresh.sum()["FP"]
    FN = Results_Thresh.sum()["FN"]
    TN = Results_Thresh.sum()["TN"]


    PPV = Results_Thresh.sum()["TP"]/(Results_Thresh.sum()["TP"] + Results_Thresh.sum()["FP"])
    FPV = Results_Thresh.sum()["FP"]/(Results_Thresh.sum()["TN"] + Results_Thresh.sum()["FP"])
    TPR = Results_Thresh.sum()["TP"]/(Results_Thresh.sum()["TP"] + Results_Thresh.sum()["FN"])
    TNR = Results_Thresh.sum()["TN"]/(Results_Thresh.sum()["TN"] + Results_Thresh.sum()["FP"])

    Precision = Results_Thresh.sum()["TP"] / (Results_Thresh.sum()["TP"] + Results_Thresh.sum()["FP"])
    Recall = Results_Thresh.sum()["TP"] / (Results_Thresh.sum()["TP"] + Results_Thresh.sum()["FN"])
    F1 = 2 * Precision * Recall / (Precision + Recall)

    # create a pandas dataframe from the confusion matrix values
    confusion_table = pd.DataFrame({'actual_positive': [1, 1, 0, 0], 'predicted_positive': [0, 1, 0, 1], 'count': [FN, TP, TN, FP]})

    # create a pivot table to represent the confusion matrix
    confusion_matrix = pd.pivot_table(confusion_table, values='count', index=['actual_positive'], columns=['predicted_positive'])

    # set the column and index names of the pivot table
    confusion_matrix.columns.name = 'Predicted Label'
    confusion_matrix.index.name = 'Actual Label'
    #confusion_matrix = confusion_matrix.T


    print("PPV={}, FPV={}\nTPR={}, TNR={}\nF1={}".format(PPV, FPV, TPR, TNR, F1))
    print(Results_Thresh.sum())
    confusion_matrix
    
    return Results_Thresh, (PPV, FPV, TPR, TNR, F1), confusion_matrix

In [ ]:
DATA_FOLDER = "../data"
COLORADO_DATA_FOLDER = os.path.join(DATA_FOLDER,"Colorado_Data")
COLORADO_DF_PATH = os.path.join(DATA_FOLDER,"colorado_outbreaks_2023-04-26.parquet")
MASTER_FIPS_PATH = os.path.join(COLORADO_DATA_FOLDER,"county_fips_master.parquet")

### Import Outbreak and Changepoint Matrices

In [ ]:
changepoint_matrix = pd.read_parquet("../data/changepoint_matrix.parquet").reset_index().rename(columns={"index": "Unnamed: 0"})
outbreak_matrix = pd.read_parquet("../data/colorado_outbreak_matrix.parquet")

In [ ]:
outbreak_matrix_df = pd.melt(outbreak_matrix, id_vars=["fips"], var_name='datetime', value_name="outbreak")
outbreak_matrix_df["fips"] = outbreak_matrix_df["fips"].astype('int64')
outbreak_matrix_df["datetime"] = pd.to_datetime(outbreak_matrix_df["datetime"])
#outbreak_matrix_df

In [ ]:
changepoint_panel_df = pd.melt(changepoint_matrix, id_vars=["Unnamed: 0"], var_name='datetime', value_name="changepoint")
changepoint_panel_df = changepoint_panel_df.fillna(0)
changepoint_panel_df = changepoint_panel_df.rename(columns={"Unnamed: 0": "datetime", "datetime": "fips"})
changepoint_panel_df["datetime"] = pd.to_datetime(changepoint_panel_df["datetime"])
changepoint_panel_df["fips"] = changepoint_panel_df["fips"].astype('int64')
changepoint_panel_df = changepoint_panel_df[changepoint_panel_df["datetime"] <= "2022-12-31"]
changepoint_panel_df.loc[changepoint_panel_df['changepoint'] < 0, 'changepoint'] = 0
#changepoint_panel_df

In [ ]:
capacity_df = pd.DataFrame(changepoint_panel_df[changepoint_panel_df["changepoint"] > 0].groupby("datetime")["changepoint"].sum())
capacity_df.index = pd.to_datetime(capacity_df.index)
capacity_df = capacity_df.resample('1D').asfreq()
capacity_df = capacity_df.fillna(0)
capacity_df = capacity_df.reset_index()
capacity_df = capacity_df.rename(columns={"changepoint":"capacity"})

merged_changepoint_outbreak_df = pd.merge(changepoint_panel_df, outbreak_matrix_df, on=["datetime","fips"])
merged_changepoint_outbreak_df = pd.merge(merged_changepoint_outbreak_df, capacity_df, on=["datetime"], how="left")
merged_changepoint_outbreak_df["capacity"] = merged_changepoint_outbreak_df["capacity"].fillna(0)
merged_changepoint_outbreak_df

In [ ]:
capacity_df

In [ ]:
DP_dates = sorted(merged_changepoint_outbreak_df[merged_changepoint_outbreak_df["capacity"] > 0]["datetime"].unique())
merged_changepoint_outbreak_df[merged_changepoint_outbreak_df["datetime"].isin(DP_dates)]

In [ ]:
merged_changepoint_outbreak_df[merged_changepoint_outbreak_df["changepoint"] < 0]

In [ ]:
merged_changepoint_outbreak_df[(merged_changepoint_outbreak_df["datetime"]=="2020-03-19") & (merged_changepoint_outbreak_df["changepoint"] == 1)]

### Import Colorado Historical Data

In [ ]:
# ── HPC-ONLY CELL ─────────────────────────────────────────────────────────
# This cell merges raw HPC outputs that are not included in the submission.
# The merged output (cdphe_tlgrf_historical_unfiltered.parquet) is already
# provided in case_study/data/ — skip this cell and continue from cell below.
# ──────────────────────────────────────────────────────────────────────────
### Load in Combined Census Data
# augmented_data_path = "../../data/augmented_us-counties-states_latest.csv"
#augmented_df = (dd.read_csv(augmented_data_path, assume_missing=True)).compute()
# augmented_df = (dd.read_csv(augmented_data_path, assume_missing=True))
# augmented_colorado_df = augmented_df[augmented_df["state"]=="Colorado"]
# augmented_colorado_df = augmented_colorado_df.compute()
# augmented_colorado_df = augmented_colorado_df.sort_values(by=["date","fips"])
# cols_to_drop = augmented_colorado_df.columns[augmented_colorado_df.nunique()==1]
# reduced_augmented_colorado_df = augmented_colorado_df.drop(cols_to_drop, axis=1)
# reduced_augmented_colorado_df


In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

### Import TLGRF Decisions

In [ ]:
pass

In [ ]:
pass

In [ ]:
# Load pre-computed merged outputs
CDPHE_w_TLGRF_merged = pd.read_parquet("../data/cdphe_tlgrf_historical_unfiltered.parquet")
CDPHE_w_TLGRF_merged["date"] = pd.to_datetime(CDPHE_w_TLGRF_merged["date"])
CDPHE_w_TLGRF_merged["date.x"] = pd.to_datetime(CDPHE_w_TLGRF_merged["date.x"])
filtered_CDPHE_w_TLGRF_merged = pd.read_parquet("../data/cdphe_tlgrf_historical_filtered.parquet")
filtered_CDPHE_w_TLGRF_merged["date"] = pd.to_datetime(filtered_CDPHE_w_TLGRF_merged["date"])
filtered_CDPHE_w_TLGRF_merged["date.x"] = pd.to_datetime(filtered_CDPHE_w_TLGRF_merged["date.x"])
old_filtered_CDPHE_w_TLGRF_merged = filtered_CDPHE_w_TLGRF_merged[filtered_CDPHE_w_TLGRF_merged["date.x"] <= "2021-09-30"]

In [ ]:
CDPHE_w_TLGRF_merged[CDPHE_w_TLGRF_merged["capacity"] < 0]

In [ ]:
CDPHE_w_TLGRF_merged.groupby("date")["changepoint"].sum()

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

### Updated Dataset

In [ ]:
pass

In [ ]:
binary_classification_performance(filtered_CDPHE_w_TLGRF_merged, outcome_col="delta_ranked", predictor_col="TLGRF_decision")

### Improvement in TP

In [ ]:
(107-33)/33

### Improvement in TN

In [ ]:
0.9643705463182898 - 0.9292161520190024

### Add Nurse and Inmate Data

In [ ]:
### Add Nurse and Inmate Data
NURSE_CORRECTION_FOLDER = os.path.join(DATA_FOLDER,"matching")

county_fips_master_df = CDPHE_w_TLGRF_merged[["fips","county_x"]].drop_duplicates()
county_fips_master_df["county_name"] = county_fips_master_df["county_x"] + " County"
county_fips_master_df = county_fips_master_df.drop(columns=["county_x"])
county_fips_master_df

nursing_ts_df = pd.read_parquet(os.path.join(NURSE_CORRECTION_FOLDER,"ReplicationData_TimeSeries_CO.parquet"))
nursing_cross_df = pd.read_parquet(os.path.join(NURSE_CORRECTION_FOLDER,"ReplicationData_CrossSection.parquet"))
nursing_cross_df = nursing_cross_df[nursing_cross_df["providerstate"]=="CO"].sort_values(by="Provider_ID")
nursing_CO_df = pd.merge(left=nursing_ts_df, right=nursing_cross_df, how="left", left_on="PROVNAME", right_on="providername")
nursing_col_keep = ["COUNTY", 'PROVNUM', 'PROVNAME',"numberofallbeds"]
nursing_CO_use_df = nursing_CO_df[nursing_col_keep]
nursing_CO_use_df = pd.merge(left=county_fips_master_df[["fips","county_name"]],right=nursing_CO_use_df,how="left",left_on="county_name",right_on="COUNTY")

nursing_CO_use_df = nursing_CO_use_df.drop(["county_name"],axis=1)

nursing_CO_use_df = nursing_CO_use_df.drop_duplicates()
nursing_CO_use_df = nursing_CO_use_df[~nursing_CO_use_df["numberofallbeds"].isna()]


prisons_df = pd.read_parquet(os.path.join(NURSE_CORRECTION_FOLDER,"facilities.parquet"))
prisons_df = prisons_df[prisons_df["facility_state"]=="Colorado"]
prisons_col_use = ["nyt_id","facility_name","facility_county_fips","max_inmate_population_2020"]
prisons_use_df = prisons_df[prisons_col_use]

beds_per_county_df = nursing_CO_use_df.groupby(["fips"])["numberofallbeds"].agg(num_beds='sum', count_homes='count').reset_index(level=0)
inmates_per_county_df = prisons_use_df.groupby("facility_county_fips")["max_inmate_population_2020"].agg(num_inmates='sum', count_facilities='count').reset_index(level=0)
beds_and_inmates_per_county_df = pd.merge(beds_per_county_df, inmates_per_county_df, left_on="fips", right_on="facility_county_fips", how="inner")
beds_and_inmates_per_county_df = beds_and_inmates_per_county_df.drop(columns="facility_county_fips")
beds_and_inmates_per_county_df[["num_beds","num_inmates","count_homes","count_facilities"]] = beds_and_inmates_per_county_df[["num_beds","num_inmates","count_homes","count_facilities"]].fillna(0)

In [ ]:
beds_filtered_CDPHE_w_TLGRF_merged = pd.merge(filtered_CDPHE_w_TLGRF_merged, beds_and_inmates_per_county_df, on="fips", how="left")
LAT_loc = beds_filtered_CDPHE_w_TLGRF_merged.columns.get_loc("LAT")
log_rolled_cases_loc = beds_filtered_CDPHE_w_TLGRF_merged.columns.get_loc("log_rolled_cases")
num_beds_loc = beds_filtered_CDPHE_w_TLGRF_merged.columns.get_loc("num_beds")


col_reshuffled = list(beds_filtered_CDPHE_w_TLGRF_merged.copy())
col_reshuffled = col_reshuffled[:(log_rolled_cases_loc)] + col_reshuffled[num_beds_loc:] + col_reshuffled[log_rolled_cases_loc:num_beds_loc]

beds_filtered_CDPHE_w_TLGRF_merged = beds_filtered_CDPHE_w_TLGRF_merged[col_reshuffled]
beds_filtered_CDPHE_w_TLGRF_merged[["num_beds","num_inmates","count_homes","count_facilities"]] = beds_filtered_CDPHE_w_TLGRF_merged[["num_beds","num_inmates","count_homes","count_facilities"]].fillna(0)

### CDPHE_TP but TLGRF_FP

In [ ]:
CDPHE_TP_TLGRF_FP_mask = (beds_filtered_CDPHE_w_TLGRF_merged["CDPHE_TP"] == True) & (beds_filtered_CDPHE_w_TLGRF_merged["TLGRF_TP"] == False)
CDPHE_TP_TLGRF_FP = beds_filtered_CDPHE_w_TLGRF_merged[CDPHE_TP_TLGRF_FP_mask]
CDPHE_TP_TLGRF_FP = CDPHE_TP_TLGRF_FP.sort_values(by=["datetime_x","fips"])
CDPHE_TP_TLGRF_FP

### TLGRF_TP but CDPHE_FP

In [ ]:
CDPHE_FP_TLGRF_TP_mask = (beds_filtered_CDPHE_w_TLGRF_merged["CDPHE_TP"] == False) & (beds_filtered_CDPHE_w_TLGRF_merged["TLGRF_TP"] == True)
CDPHE_FP_TLGRF_TP = beds_filtered_CDPHE_w_TLGRF_merged[CDPHE_FP_TLGRF_TP_mask]
CDPHE_FP_TLGRF_TP = CDPHE_FP_TLGRF_TP.sort_values(by=["datetime_x","fips"])
display(CDPHE_FP_TLGRF_TP)

In [ ]:
CDPHE_right_TLGRF_wrong_fips = CDPHE_TP_TLGRF_FP["fips"].unique()
CDPHE_right_TLGRF_wrong_county = CDPHE_TP_TLGRF_FP["county_x"].unique()


In [ ]:
set(CDPHE_right_TLGRF_wrong_county)-set(CDPHE_FP_TLGRF_TP[CDPHE_FP_TLGRF_TP["fips"].isin(CDPHE_right_TLGRF_wrong_fips)]["county_x"].unique())

In [ ]:
CDPHE_TP_TLGRF_FP.describe()

In [ ]:
CDPHE_FP_TLGRF_TP.describe()

### t-test

In [ ]:
def t_test_results(df1, df2, start_loc=2, end_loc=-1):
    t_test_results = pd.DataFrame(columns=['Column', 't-statistic', 'p-value', 'Standard Error', 'Confidence Interval'])

    for column in df1.columns[start_loc:end_loc]:
        t_statistic, p_value = ttest_ind(df1[column], df2[column])
        mean1, mean2 = df1[column].mean(), df2[column].mean()
        std1, std2 = df1[column].std(), df2[column].std()
        n1, n2 = len(df1[column]), len(df2[column])

        standard_error = np.sqrt((std1 ** 2 / n1) + (std2 ** 2 / n2))
        confidence_interval = (mean1 - mean2) - (1.96 * standard_error), (mean1 - mean2) + (1.96 * standard_error)


        if not math.isnan(t_statistic) and not math.isinf(t_statistic):
            t_test_results = pd.concat([t_test_results, pd.DataFrame([{
            'Column': column,
            't-statistic': t_statistic,
            'p-value': p_value,
            'Standard Error': standard_error,
            'Confidence Interval': confidence_interval
        }])], ignore_index=True)
            #print("Column:", column)
            #print("T-statistic:", t_statistic)
            #print("P-value:", p_value)
            #print()
    return t_test_results

In [ ]:
LAT_loc = beds_filtered_CDPHE_w_TLGRF_merged.columns.get_loc("LAT")
log_rolled_cases_loc = beds_filtered_CDPHE_w_TLGRF_merged.columns.get_loc("log_rolled_cases")

In [ ]:
CDPHE_TP = beds_filtered_CDPHE_w_TLGRF_merged[beds_filtered_CDPHE_w_TLGRF_merged["CDPHE_TP"] == 1]
TLGRF_FP = beds_filtered_CDPHE_w_TLGRF_merged[beds_filtered_CDPHE_w_TLGRF_merged["TLGRF_FP"] == 1]
CDPHE_TP_TLGRF_FP_t_test = t_test_results(CDPHE_TP, TLGRF_FP, start_loc=LAT_loc, end_loc=log_rolled_cases_loc)
CDPHE_TP_TLGRF_FP_t_test = CDPHE_TP_TLGRF_FP_t_test[CDPHE_TP_TLGRF_FP_t_test["p-value"] <= 0.10]
CDPHE_TP_TLGRF_FP_t_test

In [ ]:
CDPHE_TP_TLGRF_FP_t_test.shape

In [ ]:
CDPHE_FP = beds_filtered_CDPHE_w_TLGRF_merged[beds_filtered_CDPHE_w_TLGRF_merged["CDPHE_FP"] == 1]
TLGRF_TP = beds_filtered_CDPHE_w_TLGRF_merged[beds_filtered_CDPHE_w_TLGRF_merged["TLGRF_TP"] == 1]
CDPHE_FP_TLGRF_TP_t_test = t_test_results(CDPHE_FP, TLGRF_TP, start_loc=LAT_loc, end_loc=log_rolled_cases_loc)
CDPHE_FP_TLGRF_TP_t_test = CDPHE_FP_TLGRF_TP_t_test[CDPHE_FP_TLGRF_TP_t_test["p-value"] <= 0.10]
CDPHE_FP_TLGRF_TP_t_test

In [ ]:


display(CDPHE_FP_TLGRF_TP_t_test)

In [ ]:
CDPHE_FP_TLGRF_TP_t_test.shape

In [ ]:
CDPHE_FP_TLGRF_FP_t_test = t_test_results(CDPHE_FP, TLGRF_FP, start_loc=LAT_loc, end_loc=log_rolled_cases_loc)
CDPHE_FP_TLGRF_FP_t_test = CDPHE_FP_TLGRF_FP_t_test[CDPHE_FP_TLGRF_FP_t_test["p-value"] <= 0.10]
CDPHE_FP_TLGRF_FP_t_test

In [ ]:
CDPHE_FP.shape

In [ ]:
TLGRF_FP.shape

### CDPHE_FP, CDPHE_FN

In [ ]:
CDPHE_FP.describe()

In [ ]:
CDPHE_FP[["fips","county_x"]].value_counts()

In [ ]:
CDPHE_FN = beds_filtered_CDPHE_w_TLGRF_merged[beds_filtered_CDPHE_w_TLGRF_merged["CDPHE_FN"]==1]
CDPHE_FN.describe()

In [ ]:
CDPHE_FN[["fips","county_x"]].value_counts()

In [ ]:
CDPHE_FN

In [ ]:
CDPHE_misclassified = beds_filtered_CDPHE_w_TLGRF_merged[(beds_filtered_CDPHE_w_TLGRF_merged["CDPHE_FP"] == 1)|(beds_filtered_CDPHE_w_TLGRF_merged["CDPHE_FN"] == 1)]
CDPHE_misclassified.describe()

In [ ]:
CDPHE_misclassified_counts = pd.DataFrame(CDPHE_misclassified[["fips","county_x"]].value_counts())
CDPHE_misclassified_counts = CDPHE_misclassified_counts.rename(columns={"count":"# Times Misclassified"})
CDPHE_misclassified_counts["Cumulative"] = CDPHE_misclassified_counts["# Times Misclassified"].cumsum()
CDPHE_misclassified_counts["Cumulative Percentage"] = 100*CDPHE_misclassified_counts["Cumulative"] / CDPHE_misclassified_counts["Cumulative"].max()

CDPHE_misclassified_counts = CDPHE_misclassified_counts.reset_index()
CDPHE_misclassified_counts

In [ ]:
plt.plot(CDPHE_misclassified_counts["Cumulative"])

### TLGRF_FP

In [ ]:
TLGRF_FP.describe()

In [ ]:
TLGRF_FP[["fips","county_x"]].value_counts()

In [ ]:
TLGRF_FN = beds_filtered_CDPHE_w_TLGRF_merged[beds_filtered_CDPHE_w_TLGRF_merged["TLGRF_FN"]==1]
TLGRF_FN.describe()

In [ ]:
TLGRF_FN[["fips","county_x"]].value_counts()

In [ ]:
TLGRF_misclassified = beds_filtered_CDPHE_w_TLGRF_merged[(beds_filtered_CDPHE_w_TLGRF_merged["TLGRF_FP"] == 1)|(beds_filtered_CDPHE_w_TLGRF_merged["TLGRF_FN"] == 1)]
TLGRF_misclassified.describe()

In [ ]:
TLGRF_misclassified_counts = pd.DataFrame(TLGRF_misclassified[["fips","county_x"]].value_counts())
TLGRF_misclassified_counts = TLGRF_misclassified_counts.rename(columns={"count":"# Times Misclassified"})
TLGRF_misclassified_counts["Cumulative"] = TLGRF_misclassified_counts["# Times Misclassified"].cumsum()
TLGRF_misclassified_counts["Cumulative Percentage"] = 100*TLGRF_misclassified_counts["Cumulative"] / TLGRF_misclassified_counts["Cumulative"].max()

TLGRF_misclassified_counts = TLGRF_misclassified_counts.reset_index()
TLGRF_misclassified_counts

In [ ]:
plt.figure(figsize=(20,10))
plt.plot(TLGRF_misclassified_counts["Cumulative Percentage"], label="TLGRF")
plt.plot(CDPHE_misclassified_counts["Cumulative Percentage"], label="CDPHE")
plt.legend()
plt.xlabel("Number of Unique Misclassified Counties")
plt.ylabel("Cumulative % of Misclassifications")
plt.title("Cumulative % of Total Misclassifications against Number of Unique Misclassified Counties")

In [ ]:
plt.figure(figsize=(20,10))
plt.plot(TLGRF_misclassified_counts["Cumulative"], label="TLGRF")
plt.plot(CDPHE_misclassified_counts["Cumulative"], label="CDPHE")
plt.legend()
plt.xlabel("Number of Unique Misclassified Counties")
plt.ylabel("Cumulative # of Misclassifications")
plt.title("Cumulative # of Total Misclassifications against Number of Unique Misclassified Counties")

### Compare Top 10 Most Misclassified Counties with Everything

In [ ]:
display(CDPHE_misclassified_counts.head(10))

In [ ]:
display(TLGRF_misclassified_counts.head(10))

In [ ]:
CDPHE_w_TLGRF_merged["fips"].unique()

### "Census Data" + Nurse + Correctional Facilities

In [ ]:
beds_filtered_CDPHE_w_TLGRF_merged["fips"].unique()


LAT_loc = beds_filtered_CDPHE_w_TLGRF_merged.columns.get_loc("LAT")
log_rolled_cases_loc = beds_filtered_CDPHE_w_TLGRF_merged.columns.get_loc("log_rolled_cases")

cols_to_keep = ["fips", "county_x"] + list(beds_filtered_CDPHE_w_TLGRF_merged.columns[LAT_loc:log_rolled_cases_loc])
overall_census_data = beds_filtered_CDPHE_w_TLGRF_merged[cols_to_keep].drop_duplicates(keep="first")
#overall_census_data = pd.merge(overall_census_data, beds_and_inmates_per_county_df, on="fips", how="inner")
overall_census_data

In [ ]:
CDPHE_misclassified_census = overall_census_data[overall_census_data["fips"].isin(CDPHE_misclassified_counts.head(10)["fips"])]
CDPHE_misclassified_census

In [ ]:
TLGRF_misclassified_census = overall_census_data[overall_census_data["fips"].isin(TLGRF_misclassified_counts.head(10)["fips"])]
TLGRF_misclassified_census

In [ ]:
col_of_interest = ["E_TOTPOP", "E_POV", "E_UNEMP", "E_AGE65", "E_AGE17", "E_DISABL", "E_UNINSUR", "num_beds", "num_inmates"]

In [ ]:
CDPHE_vs_all_ttest = t_test_results(CDPHE_misclassified, beds_filtered_CDPHE_w_TLGRF_merged[beds_filtered_CDPHE_w_TLGRF_merged["P"]==1], start_loc=LAT_loc, end_loc=log_rolled_cases_loc)
CDPHE_vs_all_ttest[CDPHE_vs_all_ttest["Column"].isin(col_of_interest)].T

In [ ]:
TLGRF_vs_all_ttest = t_test_results(TLGRF_misclassified, beds_filtered_CDPHE_w_TLGRF_merged[beds_filtered_CDPHE_w_TLGRF_merged["P"]==1], start_loc=LAT_loc, end_loc=log_rolled_cases_loc)
TLGRF_vs_all_ttest[CDPHE_vs_all_ttest["Column"].isin(col_of_interest)].T

In [ ]:
CDPHE_vs_TLGRF_ttest = t_test_results(CDPHE_misclassified_census, TLGRF_misclassified_census, start_loc=2, end_loc=-1)
CDPHE_vs_TLGRF_ttest[CDPHE_vs_TLGRF_ttest["Column"].isin(col_of_interest)].T

In [ ]:
CDPHE_misclassified_summary = CDPHE_misclassified[col_of_interest].describe().replace("50%","median")
TLGRF_misclassified_summary = TLGRF_misclassified[col_of_interest].describe().replace("50%","median")
overall_misclassified_summary = (beds_filtered_CDPHE_w_TLGRF_merged[beds_filtered_CDPHE_w_TLGRF_merged["P"]==1][col_of_interest]).describe().replace("50%","median")

In [ ]:
overall_misclassified_summary

In [ ]:
stats_to_drop = ["count","std","min","max","25%","75%"]

CDPHE_selected = CDPHE_misclassified_summary.drop(stats_to_drop).replace("50%","median")
TLGRF_selected = TLGRF_misclassified_summary.drop(stats_to_drop).replace("50%","median")
overall_selected = overall_misclassified_summary.drop(stats_to_drop).replace("50%","median")



comparison_df = pd.concat([CDPHE_selected, TLGRF_selected, overall_selected], keys=['CDPHE Misclassified Counties', 'TLGRF Misclassified Counties', 'Overall'])
comparison_df.index = comparison_df.index.set_names(['Counties of:', 'Statistics'])

comparison_df

In [ ]:
comparison_df.to_latex(multirow=True, multicolumn=True)

In [ ]:
comparison_df

### CDPHE TP vs Overall

In [ ]:
CDPHE_TP_counts = pd.DataFrame(CDPHE_TP[["fips","county_x"]].value_counts())
CDPHE_TP_counts = CDPHE_TP_counts.rename(columns={"count":"# Times TP"})
CDPHE_TP_counts["Cumulative"] = CDPHE_TP_counts["# Times TP"].cumsum()
CDPHE_TP_counts["Cumulative Percentage"] = 100*CDPHE_TP_counts["Cumulative"] / CDPHE_TP_counts["Cumulative"].max()

CDPHE_TP_counts = CDPHE_TP_counts.reset_index()
CDPHE_TP_counts.head(10)

In [ ]:
TLGRF_TP_counts = pd.DataFrame(TLGRF_TP[["fips","county_x"]].value_counts())
TLGRF_TP_counts = TLGRF_TP_counts.rename(columns={"count":"# Times TP"})
TLGRF_TP_counts["Cumulative"] = TLGRF_TP_counts["# Times TP"].cumsum()
TLGRF_TP_counts["Cumulative Percentage"] = 100*TLGRF_TP_counts["Cumulative"] / TLGRF_TP_counts["Cumulative"].max()

TLGRF_TP_counts = TLGRF_TP_counts.reset_index()
TLGRF_TP_counts.head(10)

In [ ]:
CDPHE_TP_summary = CDPHE_TP[col_of_interest[:-2]].head(18).describe().replace("50%","median")
CDPHE_selected_TP = CDPHE_TP_summary.drop(stats_to_drop).replace("50%","median")

TLGRF_TP_summary = TLGRF_TP[col_of_interest[:-2]].head(74).describe().replace("50%","median")
TLGRF_selected_TP = TLGRF_TP_summary.drop(stats_to_drop).replace("50%","median") 

comparison_df_TP = pd.concat([CDPHE_selected_TP, TLGRF_selected_TP, overall_selected[col_of_interest[:-2]]], keys=['CDPHE TP Counties', "TLGRF TP Counties", 'Colorado Counties Overall'])
comparison_df_TP

In [ ]:
comparison_df_TP.to_latex(multirow=True, multicolumn=True, float_format="{:0.2f}".format)

### t-test for CDPHE vs TLGRF on Vulnerable Populations with SE and CI

In [ ]:
### Import in
restricted_col_of_interest = ["E_TOTPOP", "num_beds", "num_inmates"]
rename_dict = {"E_TOTPOP":"Total Population", "num_beds": "Nursing Home Beds", "num_inmates": "Inmates"}

CDPHE_choices = beds_filtered_CDPHE_w_TLGRF_merged[beds_filtered_CDPHE_w_TLGRF_merged["changepoint"] == 1][restricted_col_of_interest].rename(columns=rename_dict)
TLGRF_choices = beds_filtered_CDPHE_w_TLGRF_merged[beds_filtered_CDPHE_w_TLGRF_merged["TLGRF_decision"] == 1][restricted_col_of_interest].rename(columns=rename_dict)

In [ ]:
CDPHE_vs_TLGRF_means = pd.DataFrame()
CDPHE_vs_TLGRF_means["CDPHE mean"] = CDPHE_choices.mean()
CDPHE_vs_TLGRF_means["TLGRF mean"] = TLGRF_choices.mean()

CDPHE_vs_TLGRF_means = CDPHE_vs_TLGRF_means.reset_index()
CDPHE_vs_TLGRF_means = CDPHE_vs_TLGRF_means.rename(columns={"index":"Column"})
CDPHE_vs_TLGRF_means

In [ ]:
CDPHE_vs_TLGRF_ttest = t_test_results(CDPHE_choices, TLGRF_choices, start_loc=0, end_loc=len(restricted_col_of_interest))
CDPHE_vs_TLGRF_ttest_w_mean = pd.merge(CDPHE_vs_TLGRF_ttest, CDPHE_vs_TLGRF_means, on="Column")
ttest_columns_reshuffled = list(CDPHE_vs_TLGRF_ttest_w_mean.columns)
ttest_columns_reshuffled = ttest_columns_reshuffled[:1] + ttest_columns_reshuffled[-2:] + ttest_columns_reshuffled[1:-2]
display(CDPHE_vs_TLGRF_ttest_w_mean)

CDPHE_vs_TLGRF_ttest_w_mean = CDPHE_vs_TLGRF_ttest_w_mean[ttest_columns_reshuffled]
CDPHE_vs_TLGRF_ttest_w_mean = CDPHE_vs_TLGRF_ttest_w_mean.T

CDPHE_vs_TLGRF_ttest_w_mean.columns = CDPHE_vs_TLGRF_ttest_w_mean.iloc[0]
CDPHE_vs_TLGRF_ttest_w_mean = CDPHE_vs_TLGRF_ttest_w_mean[1:]
CDPHE_vs_TLGRF_ttest_w_mean

In [ ]:
ttest_columns_reshuffled

In [ ]:
CDPHE_vs_TLGRF_ttest_w_mean.to_latex(multirow=True, multicolumn=True, float_format="{:0.2f}".format)

### Anecdotal Story for Rev3

In [ ]:
colorado_outbreaks = pd.read_parquet(COLORADO_DF_PATH)
colorado_outbreaks["Date Reported"] = pd.to_datetime(colorado_outbreaks["Date Reported"])
colorado_outbreaks["Date Resolved"] = pd.to_datetime(colorado_outbreaks["Date Resolved"])
colorado_outbreaks["Duration"] = (colorado_outbreaks["Date Resolved"] - colorado_outbreaks["Date Reported"]).dt.days
colorado_outbreaks = colorado_outbreaks.sort_values(by=["Date Reported", "fips"])
colorado_outbreaks.head()

In [ ]:
pass

In [ ]:
colorado_outbreaks[colorado_outbreaks["Date Reported"] == "2020-07-05"]

In [ ]:
colorado_outbreaks[colorado_outbreaks["Date Reported"] == "2020-07-13"]

In [ ]:
pass

In [ ]:
pass

In [ ]:
filtered_CDPHE_w_TLGRF_merged[filtered_CDPHE_w_TLGRF_merged["date"]=="2020-07-13"]

In [ ]:
colorado_outbreaks[(colorado_outbreaks["Date Reported"] == "2021-05-05")]

### Restrict Binary Classification to Most Populous Counties

In [ ]:
top100_CDPHE=binary_classification_performance_counter(filtered_CDPHE_w_TLGRF_merged, outcome_col="delta_ranked", predictor = "CDPHE")

In [ ]:
top100_TLGRF=binary_classification_performance_counter(filtered_CDPHE_w_TLGRF_merged, outcome_col="delta_ranked", predictor = "TLGRF")

In [ ]:
county_fips_master_df

In [ ]:
fips_county_pop = filtered_CDPHE_w_TLGRF_merged[["fips","county_x","E_TOTPOP"]].drop_duplicates()
sorted_fips_county_pop = fips_county_pop.sort_values(by="E_TOTPOP", ascending=False)

In [ ]:
sorted_fips_county_pop

In [ ]:
num_counties = sorted_fips_county_pop.shape[0]
num_counties

### Top 10 Percentile Performance

In [ ]:
top_10_percentile_county_by_pop = sorted_fips_county_pop.head(int(num_counties*0.1))
top_10_percentile_county_by_pop

In [ ]:
top10_filtered = filtered_CDPHE_w_TLGRF_merged[filtered_CDPHE_w_TLGRF_merged["fips"].isin(top_10_percentile_county_by_pop["fips"])]
top10_filtered

In [ ]:
top10_CDPHE=binary_classification_performance_counter(top10_filtered, outcome_col="delta_ranked", predictor="CDPHE")

In [ ]:
top10_TLGRF=binary_classification_performance_counter(top10_filtered, outcome_col="delta_ranked", predictor="TLGRF")

### Top 20 Percentile

In [ ]:
top_20_percentile_county_by_pop = sorted_fips_county_pop.head(int(num_counties*0.2))
top_20_percentile_county_by_pop

In [ ]:
top20_filtered = filtered_CDPHE_w_TLGRF_merged[filtered_CDPHE_w_TLGRF_merged["fips"].isin(top_20_percentile_county_by_pop["fips"])]
top20_filtered

In [ ]:
top20_CDPHE=binary_classification_performance_counter(top20_filtered, outcome_col="delta_ranked", predictor="CDPHE")

In [ ]:
top20_TLGRF=binary_classification_performance_counter(top20_filtered, outcome_col="delta_ranked", predictor="TLGRF")

### Top 50 Percentile

In [ ]:
top_50_percentile_county_by_pop = sorted_fips_county_pop.head(int(num_counties*0.5))
top_50_percentile_county_by_pop

In [ ]:
top50_filtered = filtered_CDPHE_w_TLGRF_merged[filtered_CDPHE_w_TLGRF_merged["fips"].isin(top_50_percentile_county_by_pop["fips"])]
top50_filtered

In [ ]:
top50_CDPHE=binary_classification_performance_counter(top50_filtered, outcome_col="delta_ranked", predictor="CDPHE")

In [ ]:
top50_TLGRF = binary_classification_performance_counter(top50_filtered, outcome_col="delta_ranked", predictor="TLGRF")

In [ ]:
top50_TLGRF[1][0]

### Compile Results for Reviewer 1

In [ ]:
Most_Populous_Binary_df = pd.DataFrame()
p_list = [10, 20, 50, 100]
n_list = [int(num_counties*p/100) for p in p_list]

Most_Populous_Binary_df["Top p Percentile"] = p_list
Most_Populous_Binary_df["Top n Counties"] = n_list


Most_Populous_Binary_df["CDPHE PPV"] = [top10_CDPHE[1][0],top20_CDPHE[1][0],top50_CDPHE[1][0],top100_CDPHE[1][0]]
Most_Populous_Binary_df["TLGRF PPV"] = [top10_TLGRF[1][0],top20_TLGRF[1][0],top50_TLGRF[1][0],top100_TLGRF[1][0]]

Most_Populous_Binary_df["CDPHE F1"] = [top10_CDPHE[1][-1],top20_CDPHE[1][-1],top50_CDPHE[1][-1],top100_CDPHE[1][-1]]
Most_Populous_Binary_df["TLGRF F1"] = [top10_TLGRF[1][-1],top20_TLGRF[1][-1],top50_TLGRF[1][-1],top100_TLGRF[1][-1]]

In [ ]:
Most_Populous_Binary_df

In [ ]:
Most_Populous_Binary_df.to_csv("Most_Populous_Binary_df.csv", index=False)